# Reproducing Park, Wong-Campos et al. (2025) from dandiset 001266

Paper. Park P, Wong-Campos JD, Itkis DG, Lee BH, Qi Y, Davis HC, Antin B, Pasarkar A, Grimm JB,
Plutkis SE, Holland KL, Paninski L, Lavis LD, Cohen AE (2025). Dendritic excitations govern
back-propagation via a spike-rate accelerometer. *Nature Communications* 16, 1333.
[doi:10.1038/s41467-025-55819-9](https://doi.org/10.1038/s41467-025-55819-9)

Dataset. [DANDI:001266](https://dandiarchive.org/dandiset/001266/0.250710.0758), version
0.250710.0758, streamed from the archive. The paper cites the dandiset without a version. Two
versions are published, 0.250106.0948 and 0.250710.0758; their 84 assets have the same asset
identifiers, paths and sizes, so they hold the same files, and the most recent one is pinned
(protocol, author step 1.3).

Claims. `claims/001266/10.1038-s41467-025-55819-9.yaml`, written from the version of record
(PMC11794848) with its Supplementary Information, Peer Review File and Source Data. The Source
Data file (41467_2025_55819_MOESM10_ESM.xlsx, sha256 142a0621...0044774d), the outside input the
claims file names, maps each dandiset file to a figure panel; it is read here for that index
only.

What the dandiset holds. 84 NWB files, each one raw camera movie (Voltron2 fluorescence,
uint16 counts) with the DAQ channels recorded with it (laser modulation, EFS trigger, and for
the patch clamp files the amplifier outputs). The Source Data index names 83 of them, one per
figure panel example. The pooled samples behind most of the paper's statistics (17 neurons for
Fig. 3, 28 cells for Fig. 5b, the drug series of Figs. 6 and S16 to S18) are not in it, and it
holds no structural images, stimulus masks, regions of interest or pixel size.

| Claim | Tested here |
|---|---|
| source-data-index | tested |
| fig2a-dspike-dichotomy, fig2b-bimodal-distal-amplitudes, fig2e-f-dspike-maps, fig2h-length-constant | tested (Fig. 2 neuron) |
| fig3-subthreshold-no-dendritic-initiation, fig3-two-motifs-location-independent, fig3e-soma-third-bap, fig3d-distal-first-bap, fig3f-soma-plus-distal, fig3-predepolarization-rule | tested (the one neuron of Figs. 2, 3 and S8 in the dandiset) |
| fig1-denoising-variance | tested |
| fig5b-supralinear-auc | blocked (28 cells needed); the 10 cells in the dandiset are shown beside the Source Data |
| fig4a-step-example, fig4d-example-alternation, figs15-stochastic | tested |
| figs17e-ttx-high-dose, figs18g-ba-increases-calcium-spikes | tested |
| fig5a-plateau-collision, fig5d-single-bap-no-plateau, figs23-dendrite-drives-soma | tested (fig5d on 1 of 5 cells) |
| figs2-imaging-fidelity, figs2d-frame-rate-same-sensitivity | tested |
| fig3h, fig3g, figs12, fig3i, figs13, fig4b (two), fig4ce, figs14 (two), figs16 (two), figs17-ttx, figs18e, figs18-ni, fig5d-timing, fig5ef, fig5gh, fig6 (three), figs24, figs25, figs1, figs3, figs4, figs5, fig3-cheriff | blocked, data missing (pooled samples not deposited) |
| fig4f-g, figs20ab (simulations) | tested in simulations_park_2025.ipynb |
| figs20 (parameter sweeps), figs21 (Izhikevich model, ported to Python) | tested in simulations_park_2025.ipynb |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import dendrite as D

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("dandiset", D.DANDISET, "version", D.VERSION)

## Loading

Each movie is streamed once, in slabs of 668 frames read by parallel processes, and reduced as it
loads to a few traces: the soma (a disk of 7 pixels radius), 10-pixel segments along the apical
trunk, and the whole frame. The soma and trunk are found on the mean of the first 334 frames
(`dendrite.find_soma`, `dendrite.trace_trunk`); for files where the brightest blob in the lower
45% of the frame is a dendrite, the search is limited to a lower part of the frame (the
`SOMA_LOWER` fractions below, chosen by looking at the mean images, as the paper's regions were
drawn by hand). For the cell of Fig. 5c, whose trunk curves, the tracer searches a wider window
(`TRACE`).

In [ ]:
FIG23_CELL = ["230713", "230014", "230335", "230952", "231405", "231620"]  # Figs. 2, 3, S8
FILE_LABEL = {"230713": "D3 (index: Fig. 2)", "230014": "D1", "230335": "D2", "230952": "D4",
              "231405": "D4+soma", "231620": "soma"}
OTHER = ["180146", "174900", "001459", "181900", "184232", "185419", "145100", "152954", "154649",
         "174622", "200953", "201133", "201912", "202041"]
S22 = ["165407", "215424", "002658", "204403", "170657", "162842", "230744", "160204", "153417"]
TRACE = {k: (6, 2.0) for k in ["200953", "201133", "201912", "202041", "204403"]}  # curved trunk
SOMA_LOWER = {"174900": .7, "001459": .75, "181900": .7, "184232": .7, "185419": .7, "145100": .7,
              "152954": .7, "154649": .7, "153417": .75, "165407": .7}

cells = {}
t_load = time.time()
for key in FIG23_CELL + OTHER + S22:
    sta = None
    if key in ("230335", "230952"):  # stimulus-triggered movie for fig1-denoising-variance
        h = D.read_header(key)
        on, _ = D.stim_onsets(h)
        sta = (D.frame_of(h["cam_t"], on), 20, 170)
    cells[key] = D.cell_traces(key, soma_lower=SOMA_LOWER.get(key, .55), sta=sta, trace=TRACE.get(key, (4, 1.5)))
    print(key, cells[key]["X"].shape, f"{time.time() - t_load:.0f} s", flush=True)

## Data check

In [ ]:
rows = []
for key, o in cells.items():
    h = o["hdr"]
    on, off = D.stim_onsets(h)
    t = o["t"]
    dur = len(h["daq"]["mod488"]["x"]) / h["daq"]["mod488"]["rate"]
    rows.append(dict(file=key, experiment=h["experiment"], frames=h["shape"][0],
                     frame_px=f"{h['shape'][1]}x{h['shape'][2]}", frame_ms=round(1000 / o["fs"], 3),
                     movie_s=round(t[-1] - t[0] + 1 / o["fs"], 3), daq_s=round(dur, 3),
                     n_stim=len(on), stim_ms=round(float(np.median(off - on)) * 1000, 1),
                     first_stim_s=round(on[0], 3), soma=o["soma"],
                     trunk_um=round(o["dist"][-1] + 3.25, 0), counts_min=int(o["X"].min()),
                     counts_max=int(o["X"].max())))
info = pd.DataFrame(rows)
print(info.to_string())
print("frame interval (ms):", sorted(info.frame_ms.unique()), "-> rate", sorted((1000 / info.frame_ms).round(1).unique()), "Hz")
print("movie length minus DAQ length (s):", (info.movie_s - info.daq_s).round(3).unique())

Every movie is sampled every 1.03 ms (971 Hz), where the paper states 1 kHz; the movie spans
the same time as the DAQ record (to the frame). Fluorescence is in raw camera counts. Pixel size
is not in the files; 0.65 µm per pixel is the Methods' (10 pixels = 6.5 µm with the 10x
objective). With it the traced lines from the soma run 310 to 660 µm, which is consistent with
the apical dendrites of CA1 pyramidal cells, not a proof of the scale.

Clock check: the fluorescence at the soma must respond to the optogenetic stimulus at the
DAQ's onset times and not at times shifted away from them. The response is the mean somatic
-dF/F over the stimulus (at most 1 s) after each onset minus the 10 ms before, at the true
onsets and at onsets shifted by half the period (trains of 10 or more stimuli) or to before the stimulus and
after its end (single stimuli).

In [ ]:
rows = []
for key, o in cells.items():
    on, off = D.stim_onsets(o["hdr"])
    Y = D.bleach_correct(o["X"][:, 0], o["t"])
    fr = D.frame_of(o["t"], on)
    win = min(1000.0, float(np.median(off - on)) * 1000 + 10)  # the stimulus, at most 1 s
    if len(on) >= 10:  # trains: shift by half the period, into the gaps
        sh_before, sh_after = -np.median(np.diff(on)) * 500, np.median(np.diff(on)) * 500
    else:
        sh_before, sh_after = -(win + 100), win + 200
    true, before = D.stim_contrast(Y, fr, o["fs"], shift_ms=sh_before, win_ms=(0, win))
    _, after = D.stim_contrast(Y, fr, o["fs"], shift_ms=sh_after, win_ms=(0, win))
    rows.append(dict(file=key, window_ms=round(win), response_true=round(true * 100, 2),
                     shifted_before=round(before * 100, 2), shifted_after=round(after * 100, 2)))
clock = pd.DataFrame(rows)
print("somatic -dF/F response (%) in the stimulus window at the onsets and at onsets shifted away")
print(clock.to_string())
print("files where the true response is the largest:",
      int((clock.response_true > clock[["shifted_before", "shifted_after"]].max(1)).sum()), "of", len(clock))

### In 28 of 29 files the somatic response is larger at the true onsets than at the shifted ones.
In the TTX 100 nM ramp file (184232), where TTX leaves few somatic spikes, the response before
the ramp is as large as during it, so the clock could not be established there by this check.

### Recordings deposited twice

Three subjects have two files each. Their first 50 frames are compared.

In [ ]:
dups = {}
for a in D.assets():
    dups.setdefault(a["path"].split("/")[0], []).append(a)
for subj, lst in dups.items():
    if len(lst) > 1:
        frames = []
        for a in lst:
            f = D.open_nwb(D.s3_url(a))
            frames.append((a["path"].split("/")[1], a["size"], f["acquisition/camera_1/data"].shape,
                           f["acquisition/camera_1/data"][:50].astype(np.int64)))
            f.close()
        same = np.array_equal(frames[0][3], frames[1][3])
        print(subj)
        for name, size, shape, _ in frames:
            print("   ", name, size, shape)
        print("    first 50 frames identical:", same)

## source-data-index

> The dandiset's files are the recordings the paper's Source Data index assigns to figure
> panels, so that example panels can be found by file name.

The index is the sheet "Raw data in Dandi_Archive" of the Source Data file. Names are matched
after the index's spelling (underscores, spaces, stray quotes) is put into DANDI's (hyphens).
The experiment label the index gives each file is compared with the file's own
`experiment_description`.

In [ ]:
SOURCE_DATA = ("https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-55819-9/"
               "MediaObjects/41467_2025_55819_MOESM10_ESM.xlsx")
SOURCE_DATA_SHA256 = "142a0621e8581da59891f5a9e0b9f86983b5c2f080d6fba617eb02130044774d"
import hashlib
import io

import requests

blob = requests.get(SOURCE_DATA, timeout=60).content
print("sha256 matches the claims file:", hashlib.sha256(blob).hexdigest() == SOURCE_DATA_SHA256)
index = pd.read_excel(io.BytesIO(blob), sheet_name="Raw data in Dandi_Archive", header=1)
index = index.dropna(subset=["File name on DandiArchive"])
folders = {a["path"].split("/")[0][4:] for a in D.assets()}
index["dandi_name"] = index["File name on DandiArchive"].map(D.norm_name)
index["found"] = index.dandi_name.isin(folders)
print("index rows:", len(index), "found:", int(index.found.sum()), "dandiset subject folders:", len(folders),
      "files:", len(D.assets()))
print("not found:", index.loc[~index.found, "File name on DandiArchive"].tolist())
print("folders not in the index:", sorted(folders - set(index.dandi_name)))
# experiment labels: the file's experiment_description against the index's Experiment column
desc = {}
for key, o in cells.items():
    desc[D.norm_name(o["hdr"]["path"].split("/")[0])] = o["hdr"]["experiment"]
lab = index[index.dandi_name.isin(desc)].copy()
lab["file_experiment"] = lab.dandi_name.map(desc)
lab["same_label"] = [a.replace(" ", "") [:12] == b.replace(" ", "")[:12] for a, b in zip(lab.Experiment, lab.file_experiment)]
print(lab[["Experiment", "file_experiment", "same_label"]].to_string())

The criterion also identifies the files of Fig. 2 and Fig. 3 by their counts (54 trials each;
Fig. 2, 8 dSpikes; Fig. 3 D2, 49 -dSpikes and 35 +dSpikes; soma, 161 and 6; Soma + D2, 116 and
54; Fig. S8a, 80 and 12; Fig. S8c-d D4, 85 and 11). bAPs and dSpikes are counted in every file
of the neuron as in the claims file's shared method: somatic spikes within 60 ms of each
stimulus onset, high-pass reading "median" or "butter", somatic threshold 5 robust s.d. (the
claims file's standard choice) or 4 (a variant, see below), dSpike threshold at the first
density minimum above the failure mode of the distal amplitudes with FWHM < 5 ms, distal region
the traced trunk beyond 300 µm (claims file) or 300 to 400 µm (variant; beyond about 400 µm the
traced line leaves the trunk, see the profile below).

In [ ]:
calls = {}
for key in FIG23_CELL:
    o = cells[key]
    on, _ = D.stim_onsets(o["hdr"])
    for rd in D.READINGS:
        calls[(key,) + rd] = D.calls(o, on, *rd)
rows = []
for (key, r, k, hi), df in calls.items():
    per = df.groupby("trial").size().reindex(range(54), fill_value=0)
    rows.append(dict(file=key, label=FILE_LABEL[key], reading=D.reading_name(r, k, hi), trials=54,
                     bAPs=len(df), minus_dS=int((~df.dspike).sum()), plus_dS=int(df.dspike.sum()),
                     trials_with_dS=df[df.dspike].trial.nunique(),
                     bAPs_per_trial=dict(per.value_counts().sort_index()),
                     threshold=round(df.attrs["dspike_threshold"] * 100, 3)))
counts = pd.DataFrame(rows)
print(counts.to_string())
legend = {"Fig. 3 D2": (49, 35), "Fig. 3 soma": (161, 6), "Fig. 3 Soma + D2": (116, 54),
          "Fig. S8a": (80, 12), "Fig. S8c-d D4": (85, 11)}
print("legend totals (bAPs):", {k: sum(v) for k, v in legend.items()})

**Result: reproduced.** All 83 rows of the index name a file in the dandiset (81 distinct names, which are the 81 subject folders; two files are listed for two panels each). The experiment label of each file opened matches the index. The legend's counts identify the Fig. 3 files: the legend's D2 (49 + 35 = 84 bAPs) is the file named D4 (84 bAPs under the claims file's settings, 85 to 88 with the 4 s.d. threshold), and the Fig. 2 entry (file D3) gives the legend's 8 dSpikes. Three recordings are deposited twice (identical sizes and first frames).

The counts above identify the files. The legend's D2 of Fig. 3 (49 + 35 = 84 bAPs) is the file
named D4 (84 bAPs under the claims file's settings); the legend's soma and Soma + D2 are the
files named soma and D4+soma (161 to 167 and 165 to 170 bAPs with the 4 s.d. variant, 139 to 150
with 5 s.d.). The somatic threshold matters in these two burst files: at 5 robust s.d. some
closely spaced spikes fall below it, and the totals of the legend (167, 170) are met only at 4.
The index entry for Fig. 2 (file D3) gives 8 dSpikes, as its legend does.

### Where the distal signal is

Mean peak -dF/F per trunk segment for bAPs with and without dSpikes (claims file settings), in
three files. The difference between the classes is largest from 300 to about 370 µm; beyond
about 400 µm the traced line runs into dim tuft and background, so the variant region stops at
400 µm.

In [ ]:
def class_profiles(key, reading=("median", 5.0, np.inf)):
    o = cells[key]
    on, _ = D.stim_onsets(o["hdr"])
    df = calls[(key,) + reading]
    Y = D.bleach_correct(o["X"], o["t"])
    amp, ref = D.segment_amplitudes(Y[:, 1:-1], df, D.frame_of(o["t"], on), o["fs"])
    soma_amp, soma_ref = D.segment_amplitudes(Y[:, :1], df, D.frame_of(o["t"], on), o["fs"])
    return df, amp, ref, soma_amp[:, 0], soma_ref[:, 0]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), sharey=True)
for ax, key in zip(axes, ["230713", "230952", "231405"]):
    df, amp, ref, _, _ = class_profiles(key)
    d = cells[key]["dist"]
    for flag, c in [(False, "C0"), (True, "C3")]:
        ax.plot(d, amp[df.dspike.values == flag].mean(0) * 100, c, label=f"{'+' if flag else '-'}dSpike (n={int((df.dspike == flag).sum())})")
    ax.axvspan(300, 400, color="0.9")
    ax.set_title(f"{key} {FILE_LABEL[key]}")
    ax.set_xlabel("distance along traced trunk (µm)")
    print(key, "range of plotted -dF/F (%):", round(float(amp.mean(0).min() * 100), 2), "to",
          round(float(max(amp[df.dspike.values].mean(0).max(), amp[~df.dspike.values].mean(0).max()) * 100), 2))
axes[0].set_ylabel("peak -dF/F (%)")
axes[0].legend()
plt.tight_layout()
plt.show()

### Where the regions lie

The regions are found on the mean image (see Loading). The figure and table below show them for
the files the claims use, so that the soma, the traced line and the distance marks can be checked
against the image.

In [ ]:
# Where the regions lie: the mean image of each file with the traced line (cyan), the soma (yellow
# dot) and the points 200 (green), 300 (yellow) and 400 µm (red) along the line. The table gives
# their pixel coordinates (row, column) and the mean-image brightness along the line in 100 µm
# bins over the image's median (background is about 1; the trunk is brighter).
roi_keys = ["230713", "230952", "231405", "180146", "174900", "001459", "181900", "145100", "174622", "160204"]
fig, axes = plt.subplots(1, len(roi_keys), figsize=(2.0 * len(roi_keys), 4))
rows = []
for ax, key in zip(axes, roi_keys):
    o = cells[key]
    p = o["path"]
    dpath = np.r_[0, np.cumsum(np.hypot(*np.diff(p, axis=0).T))] * D.UM_PER_PX
    ax.imshow(o["mean"], cmap="gray", vmax=np.percentile(o["mean"], 99.5))
    ax.plot(p[:, 1], p[:, 0], "c-", lw=0.6)
    ax.plot(o["soma"][1], o["soma"][0], "yo", ms=4)
    row = dict(file=key, soma_rc=o["soma"], line_um=round(float(dpath[-1])))
    for um, c in [(200, "g"), (300, "y"), (400, "r")]:
        if dpath[-1] >= um:
            r_, c_ = p[np.searchsorted(dpath, um)]
            ax.plot(c_, r_, c + "x", ms=6)
            row[f"at_{um}um"] = (int(r_), int(c_))
    bright = o["mean"][p[:, 0], p[:, 1]] / np.median(o["mean"])
    for lo in range(0, 700, 100):
        sel = (dpath >= lo) & (dpath < lo + 100)
        if sel.any():
            row[f"F/bg_{lo}-{lo + 100}"] = round(float(bright[sel].mean()), 2)
    ax.set_title(key, fontsize=8)
    ax.axis("off")
    rows.append(row)
plt.tight_layout()
plt.show()
print(pd.DataFrame(rows).to_string())

The traced line runs along the bright trunk to about 400 µm: in the Fig. 2 and 3 files the mean image along it is 1.3 to 2.0 times the image median up to 400 µm and 0.73 to 0.89 beyond, darker than the median, where the line has left the dendrite. In the other files shown the ratio beyond 500 µm is 0.77 to 1.15. The region "beyond 300 µm" therefore averages trunk (300 to 400 µm) with background beyond it, which dilutes distal amplitudes; the 300 to 400 µm variant keeps only the part on the trunk.

## fig2a-dspike-dichotomy

> In the Fig. 2 neuron, stimulation of proximal oblique dendrites evoked two somatic spikes per
> trial; in most trials the distal dendrites showed only an attenuated copy, and in 8 of 54
> trials a single rapid dSpike, always on the second somatic spike.

File D3 (index entry "branch_stim (Fig.2)"). The criterion: the count of trials with a dSpike
within 8 ± 0.8 under both filter readings, and no dSpike on the first somatic spike of a trial.

In [ ]:
rows = []
for rd in D.READINGS:
    df = calls[("230713",) + rd]
    per = df.groupby("trial").size().reindex(range(54), fill_value=0)
    rows.append(dict(reading=D.reading_name(*rd), trials_with_dS=df[df.dspike].trial.nunique(),
                     dS_on_first=int(df[df.dspike & (df.order == 0)].shape[0]),
                     dS_by_order=dict(df[df.dspike].order.value_counts().sort_index()),
                     trials_2_spikes=int((per == 2).sum()), trials_3_spikes=int((per == 3).sum()),
                     dS_on_last_spike=int(sum(r.order == per[r.trial] - 1 for r in df[df.dspike].itertuples()))))
fig2a = pd.DataFrame(rows)
print(fig2a.to_string())
main = fig2a[fig2a.reading.str.contains("k=5, distal >300")]
print("claims file readings (median and butter, k=5, >300 µm): trials with a dSpike",
      main.trials_with_dS.tolist(), "; dSpikes on a first spike", main.dS_on_first.tolist())

**Result: reproduced.** With the claims file's settings and with the 4 s.d. somatic threshold, 8 of 54 trials carry a dSpike under both filter readings, and no dSpike follows the first somatic spike of a trial. The statement says more than the criterion tests: 7 to 10 of the 54 trials have three somatic spikes, not two, and most dSpikes (5 to 7 of 8) ride on the third spike; every dSpike is on the last spike of its trial. With the distal region cut at 400 µm the Butterworth reading gives 6.

## fig2b-bimodal-distal-amplitudes

> Spike amplitudes have a unimodal distribution at the soma and a bimodal one in the distal
> dendrite.

Amplitudes are the high-passed -dF/F at each bAP's peak, at the soma and in the distal region,
for the Fig. 2 file. Reading (a): Hartigan's dip test, p < 0.05 distal and p >= 0.05 soma.
Reading (b): a two-component Gaussian mixture preferred by BIC distally and not at the soma.

In [ ]:
import diptest
from sklearn.mixture import GaussianMixture

def bic_gain(x, seed=0):
    x = np.asarray(x, float).reshape(-1, 1)
    b1 = GaussianMixture(1, random_state=seed).fit(x).bic(x)
    b2 = GaussianMixture(2, random_state=seed, n_init=5).fit(x).bic(x)
    return b1 - b2  # > 0: two components preferred

rows = []
for rd in D.READINGS:
    df = calls[("230713",) + rd]
    for site in ["soma", "distal"]:
        x = df[site].values
        dip, p = diptest.diptest(x)
        rows.append(dict(reading=D.reading_name(*rd), site=site, n=len(x), dip=round(dip, 4), dip_p=round(p, 4),
                         bic1_minus_bic2=round(bic_gain(x), 1)))
fig2b = pd.DataFrame(rows)
print(fig2b.to_string())
fig, axes = plt.subplots(1, 2, figsize=(9, 2.8))
df = calls[("230713", "median", 5.0, np.inf)]
for ax, site in zip(axes, ["soma", "distal"]):
    ax.hist(df[site] * 100, 30)
    ax.set_title(f"{site} (range {df[site].min() * 100:.2f} to {df[site].max() * 100:.2f}%)")
    ax.set_xlabel("high-passed -dF/F at the bAP (%)")
axes[1].axvline(df.attrs["dspike_threshold"] * 100, color="k", ls=":")
plt.tight_layout()
plt.show()

**Result: blocked (methods underspecified).** The two readings disagree. The Gaussian mixture prefers two components for the distal amplitudes (BIC lower by 55 to 125) and one at the soma (BIC higher by 11 to 12), as the claim says; the dip test finds no bimodality anywhere (distal p 0.63 to 0.99, soma p 0.70 to 0.93), as 8 large events among about 115 are too few for it. The criterion makes this methods_underspecified.

## fig2e-f-dspike-maps

> bAPs without a dSpike are confined to the soma and proximal dendrites, while bAPs with a
> dSpike are amplified in the distal dendrites.

Per trunk segment, the mean peak -dF over bAPs of each class is divided by the segment's
reference amplitude (mean -dF from 10 to 20 ms after the spike, Fig. 1d legend; the reference is
averaged over all bAPs of the file, as it describes the segment, not the class). The soma value
is the same ratio in the somatic disk; the distal value treats the distal segments as one region
(mean peak over mean reference). Amplitudes are relative to the trial's pre-stimulus baseline.
Criterion: the -dSpike average beyond 300 µm below half its soma value, and the +dSpike average
beyond 300 µm at least its soma value. The reference, measured 10 to 20 ms after each spike,
includes the depolarization the stimulus drives near the stimulated branch, so the ratio
without it (raw) is shown as a variant.

In [ ]:
rows = []
for rd in D.READINGS:
    df, amp, ref, samp, sref = class_profiles("230713", rd)
    d = cells["230713"]["dist"]
    region = (d > 300) & (d < rd[2])
    out = dict(reading=D.reading_name(*rd))
    for flag, name in [(False, "minus"), (True, "plus")]:
        sel = df.dspike.values == flag
        soma = samp[sel].mean() / sref.mean()
        distal = amp[sel][:, region].mean() / ref[:, region].mean()  # the region as one ROI
        out[f"{name}_soma"] = round(soma, 2)
        out[f"{name}_distal"] = round(float(distal), 2)
        out[f"{name}_ratio"] = round(float(distal / soma), 2)
        out[f"{name}_raw_ratio"] = round(float(amp[sel][:, region].mean() / samp[sel].mean()), 2)
    rows.append(out)
fig2ef = pd.DataFrame(rows)
print("dF/F_ref at the soma and over the distal region (one ROI); ratio = distal / soma;")
print("raw_ratio: the same without the reference (peak -dF/F, a variant)")
print(fig2ef.to_string())

**Result: not reproduced.** bAPs with a dSpike are larger in the distal region than bAPs without (dF/F_ref 3.3 against 2.1 beyond 300 µm; 2.6 to 2.8 against 1.5 between 300 and 400 µm), but no reading meets both of the criterion's thresholds. With dF/F_ref, the -dSpike distal value is 0.72 of the soma's beyond 300 µm, not below half (0.50 to 0.51 between 300 and 400 µm, where the +dSpike value is then 0.81 to 0.86, below the soma's); beyond 300 µm the +dSpike value is 1.01 to 1.06 of the soma's. With raw -dF/F the -dSpike ratio is 0.27 to 0.44 (confined) and the +dSpike ratio 0.38 to 0.74 (below the soma).

## fig2h-length-constant

> The peak amplitude of bAPs without a dSpike decays exponentially along the apical trunk with a
> length constant of 255 ± 10 µm (mean ± s.e.m., 93 spikes).

One fit per -dSpike bAP of -dF/F_ref along the segments, then the mean of the length constants.
Readings: spikes (a) of the Fig. 2 file, (b) of all six files of the neuron; fit (a)
A exp(-x/λ), (b) A exp(-x/λ) + c. The fit runs over the traced trunk up to 400 µm (beyond it the
line leaves the trunk, see the profile above). Fits
that fail or give λ above 5000 µm (no decay) are counted and left out. Raw -dF/F (without the
reference) is a variant.

In [ ]:
def length_constants(keys, reading=("median", 5.0, np.inf), max_um=400.0, normalized=True):
    out = {False: [], True: []}
    for key in keys:
        df, amp, ref, _, _ = class_profiles(key, reading)
        d = cells[key]["dist"]
        r = ref.mean(0) if normalized else np.ones(len(d))
        sel = d < max_um
        for a in amp[~df.dspike.values]:
            y = a[sel] / r[sel]
            for off in (False, True):
                out[off].append(D.exp_fit(d[sel], y, offset=off))
    return {k: np.array(v) for k, v in out.items()}

rows = []
for spikes, keys in [("(a) Fig. 2 file", ["230713"]), ("(b) all six files", FIG23_CELL)]:
    for max_um, normalized in [(400.0, True), (400.0, False)]:
        for rd in [("median", 5.0, np.inf), ("butter", 5.0, np.inf), ("median", 4.0, np.inf)]:
            lam = length_constants(keys, rd, max_um, normalized)
            for off, form in [(False, "A exp(-x/lam)"), (True, "A exp(-x/lam) + c")]:
                v = lam[off]
                ok = np.isfinite(v) & (v < 5000)
                rows.append(dict(spikes=spikes, amplitude="dF/F_ref" if normalized else "raw -dF/F",
                                 fit_range="<=400 um" if np.isfinite(max_um) else "whole line",
                                 reading=D.reading_name(*rd), form=form, n_spikes=len(v), n_used=int(ok.sum()),
                                 mean_um=round(float(v[ok].mean()), 1) if ok.any() else np.nan,
                                 sem_um=round(float(stats.sem(v[ok])), 1) if ok.sum() > 1 else np.nan,
                                 median_um=round(float(np.median(v[ok])), 1) if ok.any() else np.nan))
fig2h = pd.DataFrame(rows)
print(fig2h.to_string())

**Result: not reproduced.** No reading gives 255 ± 10 µm. Per-spike fits of dF/F_ref along the trunk fail or exceed 5000 µm for 23 to 42% of spikes and give a mean λ of 1407 to 1488 µm without an offset and 54 to 58 µm with one. Fits of raw -dF/F without an offset give 356 ± 4 µm (Fig. 2 file, mean ± s.e.m.) and 402 to 404 ± 4 µm (all six files); with an offset they mostly fail. The reference amplitude rises near the stimulated branch, which distorts the normalized profile.

## fig3-subthreshold-no-dendritic-initiation

> Subthreshold depolarization follows the stimulus location and decays smoothly outside it;
> dSpikes never initiated in the dendrites.

Distal events are found here without reference to the soma: peaks of the high-passed distal
trace above the file's dSpike threshold with FWHM < 5 ms. Each is checked for a somatic spike
whose peak precedes it by 0 to 5 ms. The dandiset holds one neuron of the 17; its six
stimulus locations are used.

In [ ]:
rows = []
for key in FIG23_CELL:
    o = cells[key]
    for rd in D.READINGS:
        df = calls[(key,) + rd]
        Y, H = D.corrected(o, rd[0])
        ev = D.independent_distal_events(H, o["dist"], df.attrs["dspike_threshold"], o["fs"], hi_um=rd[2])
        pk, _ = D.detect_spikes(H[:, 0], o["fs"], k=rd[1])
        lag = [(e - pk[pk <= e].max()) / o["fs"] * 1000 if (pk <= e).any() else np.inf for e in ev]
        lag = np.array(lag)
        on, _ = D.stim_onsets(o["hdr"])
        after_onset = [(o["t"][e] - on[on <= o["t"][e]].max()) * 1000 if (on <= o["t"][e]).any() else np.nan for e in ev]
        after_onset = np.array(after_onset)
        rows.append(dict(file=key, label=FILE_LABEL[key], reading=D.reading_name(*rd), distal_events=len(ev),
                         soma_spike_0_5ms_before=int((lag <= 5).sum()), soma_spike_5_15ms_before=int(((lag > 5) & (lag <= 15)).sum()),
                         none_within_15ms=int((lag > 15).sum()),
                         those_ms_after_onset=np.round(after_onset[lag > 15], 1).tolist()))
fig3sub = pd.DataFrame(rows)
print(fig3sub.to_string())

In [ ]:
# The distal events above with no somatic spike within 15 ms before them: the largest high-passed
# somatic value in the 15 ms before each (in robust s.d. of the somatic trace, against the
# threshold k), and the traces around the first six.
orph = []
for key in FIG23_CELL:
    o = cells[key]
    on, _ = D.stim_onsets(o["hdr"])
    for rd in [r for r in D.READINGS if not np.isfinite(r[2])]:
        df = calls[(key,) + rd]
        Y, H = D.corrected(o, rd[0])
        ev = D.independent_distal_events(H, o["dist"], df.attrs["dspike_threshold"], o["fs"])
        pk, thr = D.detect_spikes(H[:, 0], o["fs"], k=rd[1])
        xd, _ = D.distal_trace(H, o["dist"], 300)
        w = int(0.015 * o["fs"])
        for e in ev:
            prev = pk[pk <= e]
            if len(prev) and (e - prev.max()) / o["fs"] * 1000 <= 15:
                continue
            j = max(0, e - w) + int(np.argmax(H[max(0, e - w):e + 1, 0]))
            before = on[on <= o["t"][e]]
            orph.append(dict(file=key, reading=D.reading_name(*rd), frame=int(e),
                             ms_after_onset=round(float((o["t"][e] - before.max()) * 1000), 1) if len(before) else np.nan,
                             distal_pct=round(float(xd[e] * 100), 2), dspike_thr_pct=round(float(df.attrs["dspike_threshold"] * 100), 2),
                             soma_max_15ms_before_sd=round(float(H[j, 0] / (thr / rd[1])), 1), k=rd[1],
                             soma_max_ms_before=round((e - j) / o["fs"] * 1000, 1)))
orph = pd.DataFrame(orph)
print(orph.to_string() if len(orph) else "no such events")
uniq = orph.drop_duplicates(["file", "frame"]).head(6) if len(orph) else orph
if len(uniq):
    fig, axes = plt.subplots(1, len(uniq), figsize=(3 * len(uniq), 2.6), squeeze=False)
    for ax, r in zip(axes[0], uniq.itertuples()):
        o = cells[r.file]
        Y, H = D.corrected(o, "median")
        xd, _ = D.distal_trace(H, o["dist"], 300)
        w = int(0.04 * o["fs"])
        s = slice(max(0, r.frame - w), r.frame + w)
        tt = (np.arange(len(o["t"]))[s] - r.frame) / o["fs"] * 1000
        ax.plot(tt, H[s, 0] * 100, label="soma")
        ax.plot(tt, xd[s] * 100, label="distal >300 µm")
        ax.set_title(f"{r.file} frame {r.frame}", fontsize=8)
        ax.set_xlabel("ms from distal event")
        print(r.file, r.frame, "plotted range (%):", round(float(min(H[s, 0].min(), xd[s].min()) * 100), 2), "to",
              round(float(max(H[s, 0].max(), xd[s].max()) * 100), 2))
    axes[0][0].set_ylabel("high-passed -dF/F (%), median filter")
    axes[0][0].legend(fontsize=7)
    plt.tight_layout()
    plt.show()

**Result: not reproduced.** Most distal events found without reference to the soma follow a somatic spike, but not all. With the claims file's settings (5 s.d., beyond 300 µm), file D4 has 1 event (20.7 and 21.4 ms after onset) and the D4+soma file 1 (median filter, 170 ms after onset) or 2 (Butterworth, 126 and 170 ms) with no somatic spike within 15 ms before them. The table above shows why: before the D4 event the somatic trace peaks at 4.7 to 4.9 robust s.d., a spike just under the 5 s.d. threshold (the 4 s.d. threshold finds it). Before the two D4+soma events the somatic trace stays at 2.0 to 2.4 s.d., so no somatic spike precedes them; they fall 126 and 170 ms after onset, between stimuli, with distal amplitudes of 0.74 to 1.03% against thresholds of 0.57 to 0.63%. By the criterion (no dSpike without a preceding somatic spike) the claim fails on one or two small isolated distal events; whether they are dendrite-initiated spikes or noise is not settled by the outputs. The subthreshold part of the statement is not tested. Only one of the 17 neurons is in the dandiset.

## fig3-two-motifs-location-independent

> Neurons produced only two kinds of excitation, perisomatic bAPs and distal dSpikes, and neither
> spatial profile depended on which dendrite was stimulated.

Profiles of -dF/F_ref along the traced trunk (to 400 µm, see above) for each class and each of
the six stimulus locations; raw -dF/F is a variant.
Criterion: within each class every pair of locations has Pearson r >= 0.9, and the +dSpike
profile exceeds the -dSpike profile beyond 300 µm in every location.

In [ ]:
for rd in [("median", 5.0, np.inf), ("butter", 5.0, np.inf), ("median", 4.0, np.inf), ("butter", 4.0, np.inf)]:
  for normalized in (True, False):
    prof = {False: {}, True: {}}
    exceed = {}
    nmin = min(int((cells[k]["dist"] < 400).sum()) for k in FIG23_CELL)  # the traced trunk
    for key in FIG23_CELL:
        df, amp, ref, _, _ = class_profiles(key, rd)
        r = ref.mean(0) if normalized else np.ones(amp.shape[1])
        d = cells[key]["dist"]
        for flag in (False, True):
            if (df.dspike == flag).sum():
                prof[flag][key] = (amp[df.dspike.values == flag].mean(0) / r)[:nmin]
        region = d[:nmin] > 300
        exceed[key] = round(float(np.mean(prof[True][key][region] > prof[False][key][region])), 2) if key in prof[True] else None
    print("reading:", D.reading_name(*rd), "| amplitude:", "dF/F_ref" if normalized else "raw -dF/F", "| segments to", round(float(cells[FIG23_CELL[0]]["dist"][nmin - 1]), 1), "µm")
    for flag in (False, True):
        keys = list(prof[flag])
        M = np.array([prof[flag][k] for k in keys])
        R = np.corrcoef(M)
        iu = np.triu_indices(len(keys), 1)
        print(f"  {'+' if flag else '-'}dSpike profiles: {len(keys)} locations, pairwise r min {R[iu].min():.3f}, median {np.median(R[iu]):.3f}, pairs >= 0.9: {int((R[iu] >= 0.9).sum())} of {len(iu[0])}")
    print("  fraction of segments beyond 300 µm where +dSpike exceeds -dSpike, by file:", exceed)
d = cells["230713"]["dist"][:nmin]
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, flag in zip(axes, (False, True)):
    for key in prof[flag]:
        ax.plot(d, prof[flag][key], label=FILE_LABEL[key])
    ax.set_title(f"{'+' if flag else '-'}dSpike bAPs (last reading above)")
    ax.set_xlabel("distance (µm)")
axes[0].set_ylabel("-dF/F_ref")
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()
print("plotted range:", round(min(v.min() for p in prof.values() for v in p.values()), 2), "to",
      round(max(v.max() for p in prof.values() for v in p.values()), 2))

**Result: not reproduced.** With dF/F_ref along the traced trunk, the lowest pairwise r between the six stimulus locations is 0.81 to 0.83 for the -dSpike profiles (7 to 9 of 15 pairs at 0.9 or more) and 0.31 to 0.44 for the +dSpike profiles (0 of 15 pairs). With raw -dF/F, 15 of 15 -dSpike pairs and 12 to 13 of 15 +dSpike pairs reach 0.9 (lowest r 0.88 to 0.89). The +dSpike profile exceeds the -dSpike profile in 88 to 100% of segments beyond 300 µm, not in all. The two motifs are similar across locations, but not to the criterion's thresholds.

## fig3e-soma-third-bap, fig3d-distal-first-bap, fig3f-soma-plus-distal

> fig3e: In the Fig. 3 neuron, soma stimulation never evoked dSpikes on the first two bAPs and
> evoked one on the third bAP in 6 of 54 trials.
>
> fig3d: stimulation of distal dendrite D2 evoked a dSpike on the first bAP in 35 of 54 trials
> (65%).
>
> fig3f: simultaneous soma and D2 stimulation gave two quick bAPs without dSpikes and then a third
> bAP that evoked a dSpike in all 54 trials.

dSpikes per bAP position within the trial, for the soma file, the files D2 and D4 (the
readings for the legend's D2), and the D4+soma file.

In [ ]:
rows = []
for key in ["231620", "230335", "230952", "231405"]:
    for rd in D.READINGS:
        df = calls[(key,) + rd]
        by = df.groupby("order").agg(n=("dspike", "size"), dS=("dspike", "sum"))
        rows.append(dict(file=key, label=FILE_LABEL[key], reading=D.reading_name(*rd), bAPs=len(df),
                         **{f"bAP{i + 1}_dS/n": f"{int(by.dS.get(i, 0))}/{int(by.n.get(i, 0))}" for i in range(4)}))
fig3 = pd.DataFrame(rows)
print(fig3.to_string())




**Result: not reproduced.** In the D4+soma file with the 4 s.d. threshold (167 bAPs against the legend's 170) and the claims file's region beyond 300 µm, 46 third bAPs carry a dSpike (of 51 with the median filter and 52 with the Butterworth), short of 54 ± 5.4, and 2 of 108 first and second bAPs carry one (within the criterion's 10%). With the region cut at 400 µm, where the traced line is still on the trunk, the count is 51 (median) and 49 (Butterworth), within tolerance; with 5 s.d., spikes are missed and only 30 to 35 third bAPs carry one.

**Result: reproduced (on an assumption).** The file whose totals match the legend's D2 is the file named D4 (84 to 88 bAPs against 84). In it, 33 (median filter) and 32 (Butterworth) of 54 first bAPs carry a dSpike with the 4 s.d. threshold, within 35 ± 3.5; with the claims file's 5 s.d. the Butterworth reading gives 31, just outside. The file named D2 gives 0 to 3.

**Result: not reproduced.** No reading gives 6 third-bAP dSpikes with none on the first two bAPs under both filters. With the 4 s.d. threshold (which gives the legend's bAP total, 161 to 165 against 167), the median filter gives 5 of 51 third bAPs (6 with the region cut at 400 µm) and none on the first two; the Butterworth filter gives 3 (4) on the third and 2 on the second. With 5 s.d., dSpikes fall on second bAPs in both filters.

## fig3-predepolarization-rule

> dSpikes arise if and only if the distal dendrites have been depolarized for at least 15 ms
> before the bAP arrives.

Per file, the onset of distal depolarization is the first frame after stimulus onset at which
the stimulus-triggered average distal -dF/F exceeds its pre-stimulus mean by 2 s.d. Each bAP's
time after that onset is compared with 15 ms. Criterion: at least 90% of dSpikes follow 15 ms
or more of distal depolarization, and at least 90% of bAPs that follow less than 15 ms fail.

In [ ]:
for rd in D.READINGS:
    tabs = []
    for key in FIG23_CELL:
        o = cells[key]
        on, _ = D.stim_onsets(o["hdr"])
        Y = D.bleach_correct(o["X"], o["t"])
        yd, _ = D.distal_trace(Y, o["dist"], 300, rd[2])
        onset_ms, _ = D.depolarization_onset(yd, D.frame_of(o["t"], on), o["fs"])
        df = calls[(key,) + rd].copy()
        df["depolarized_ms"] = df.t_ms - onset_ms
        df["file"] = key
        df["onset_ms"] = onset_ms
        tabs.append(df)
    allb = pd.concat(tabs)
    long_ = allb.depolarized_ms >= 15
    print("reading:", D.reading_name(*rd))
    print("  distal depolarization onset (ms after stimulus) by file:",
          {k: round(float(v), 1) for k, v in allb.groupby("file").onset_ms.first().items()})
    print("  dSpikes after >= 15 ms:", int((allb.dspike & long_).sum()), "of", int(allb.dspike.sum()),
          f"({100 * (allb.dspike & long_).sum() / max(1, allb.dspike.sum()):.0f}%)")
    print("  bAPs after < 15 ms that fail:", int((~allb.dspike & ~long_).sum()), "of", int((~long_).sum()),
          f"({100 * (~allb.dspike & ~long_).sum() / max(1, (~long_).sum()):.0f}%)")
    print("  bAPs after >= 15 ms with a dSpike:", int((allb.dspike & long_).sum()), "of", int(long_.sum()))
    print(pd.crosstab([allb.file, long_.rename(">=15ms")], allb.dspike).to_string())

**Result: not reproduced.** The distal region depolarizes 2 to 5 ms after stimulus onset in every file, and only 45 to 48% of dSpikes come after 15 ms or more of distal depolarization (criterion 90%), while 87 to 89% of bAPs that come earlier fail. The early dSpikes are mostly first-bAP dSpikes under distal stimulation (file D4) and second-bAP dSpikes in the D4+soma file. The onset measure (2 s.d. above the stimulus-triggered baseline) is the claims file's standard choice; a different onset could move these numbers.

## fig1-denoising-variance

> The first principal components of the pixel waveforms hold more than 96% of their variance
> (the first five, Fig. S7b; the first three, Methods).

PCA of the stimulus-triggered average movie of the Fig. 3a recording (the legend's D2; readings
file D4 and file D2), pixel waveforms from 20 frames before to 170 after onset, each pixel's
pre-stimulus mean subtracted. Pixels: those of the cell (mean image above its 75th percentile)
and, as a variant, all pixels. Trials: all 54, and the first 11 (Fig. S7a legend's number).

In [ ]:
def pca_share(movie, mask, pre=20, how="centred"):
    """Share of variance in the first 3 and 5 components of the pixel waveforms. how: 'centred'
    (each pixel's pre-stimulus mean and the across-pixel mean waveform removed), 'baseline' (only
    each pixel's pre-stimulus mean removed, uncentred SVD), 'uncentred' (raw counts, uncentred SVD)."""
    M = movie[:, mask.ravel()].T.astype(np.float64)  # pixels x lags
    if how in ("centred", "baseline"):
        M = M - M[:, :pre].mean(1, keepdims=True)
    if how == "centred":
        M = M - M.mean(0, keepdims=True)
    s = np.linalg.svd(M, compute_uv=False) ** 2
    return s[:3].sum() / s.sum(), s[:5].sum() / s.sum()

rows = []
for key in ["230952", "230335"]:
    o = cells[key]
    on, _ = D.stim_onsets(o["hdr"])
    fr = D.frame_of(o["t"], on)
    h11 = dict(o["hdr"], shape=(int(fr[10] + 170),) + tuple(o["hdr"]["shape"][1:]))
    h11["url"] = D.s3_url(D.asset_for(key))
    _, movie11 = D.extract(h11, D.weights([np.ones(o["mean"].shape, bool)]), sta=(fr[:11], 20, 170))
    for trials, movie in [("54", o["sta_movie"]), ("first 11", movie11)]:
        for pix, mask in [("cell (top 25%)", o["mean"] > np.percentile(o["mean"], 75)), ("all", np.ones(o["mean"].shape, bool))]:
            for how in ("centred", "baseline", "uncentred"):
                s3, s5 = pca_share(movie, mask, how=how)
                rows.append(dict(file=key, label=FILE_LABEL[key], trials=trials, pixels=pix, n_pixels=int(mask.sum()),
                                 preprocessing=how, first3_pct=round(100 * s3, 1), first5_pct=round(100 * s5, 1)))
fig1 = pd.DataFrame(rows)
print(fig1.to_string())

**Result: blocked (methods underspecified).** The share depends entirely on how the pixel waveforms are prepared, which the paper does not say. With each pixel's pre-stimulus mean and the across-pixel mean waveform removed (centred), the first five components hold 22 to 30% of the variance over 54 trials and 10 to 12% over the first 11; with only each pixel's baseline removed (uncentred SVD), 39 to 58% and 16 to 24%; on raw counts (uncentred SVD of the waveforms with their resting fluorescence), 100.0% in every case, because the first component is the resting image. The bound (86.4%) is met only by the raw-count reading, which measures the resting brightness of the pixels, not their voltage waveforms. The readings give opposite outcomes and the paper gives no basis to choose one.

## Other example recordings: dSpike calls

For the step, wide-field and drug examples the bAPs are the somatic spikes within each
stimulus epoch, and dSpikes are called as above (both filters; somatic threshold 5 or 4 robust
s.d.; distal region beyond 300 µm on the traced line, or 300 to 400 µm).

In [ ]:
def seq(df, n=None):
    out = {}
    for tr, g in df.groupby("trial"):
        s = "".join("D" if x else "F" for x in g.sort_values("order").dspike)
        out[tr] = s[:n] if n else s
    return out

ocalls = {}
for key in ["180146", "174900", "001459", "181900", "184232", "185419", "145100", "152954", "154649"]:
    o = cells[key]
    on, off = D.stim_onsets(o["hdr"])
    dur = float(np.median(off - on))
    for rd in D.READINGS:
        ocalls[(key,) + rd] = D.calls(o, on, *rd, window_s=(0.0, dur))
rows = []
for (key, r, k, hi), df in ocalls.items():
    rows.append(dict(file=key, experiment=cells[key]["hdr"]["experiment"], reading=D.reading_name(r, k, hi),
                     epochs=df.trial.nunique(), bAPs=len(df), dSpikes=int(df.dspike.sum()),
                     threshold=round(df.attrs["dspike_threshold"] * 100, 3),
                     trough_found=bool(np.isfinite(df.attrs["dspike_threshold"])),
                     kde_modes=D.trough_threshold(df.distal.values)[1],
                     above_thr_but_fwhm_ge_5ms=int(((df.distal > df.attrs["dspike_threshold"]) & (df.fwhm_ms >= 5)).sum()),
                     distal_max=round(df.distal.max() * 100, 3) if len(df) else np.nan))
print(pd.DataFrame(rows).to_string())
print("Where trough_found is False the distal amplitudes have one density mode: no threshold separates dSpikes from failures,")
print("and the dSpike count of that reading is undefined (shown as 0 above, and read as undefined in the conclusions).")
fig, axes = plt.subplots(1, 4, figsize=(13, 2.6))
for ax, key in zip(axes, ["174900", "001459", "181900", "184232"]):
    df = ocalls[(key, "median", 5.0, np.inf)]
    x = df.distal.values * 100
    ax.hist(x, 40, density=True, color="0.7")
    if len(x) > 4 and np.ptp(x) > 0:
        g = np.linspace(x.min(), x.max(), 300)
        ax.plot(g, stats.gaussian_kde(x)(g), "k")
    thr = df.attrs["dspike_threshold"]
    if np.isfinite(thr):
        ax.axvline(thr * 100, color="r", ls=":")
    ax.set_title(f"{key} (median, k=5, >300 µm)", fontsize=8)
    ax.set_xlabel("distal amplitude (%)")
    print(key, "distal amplitude range (%):", round(float(x.min()), 2) if len(x) else None, "to", round(float(x.max()), 2) if len(x) else None,
          "; threshold:", round(float(thr * 100), 3) if np.isfinite(thr) else "none (one mode)")
plt.tight_layout()
plt.show()

## fig4a-step-example

> In the Fig. 4a neuron, a step stimulus to a proximal apical dendrite evoked a bAP without a
> dSpike, then two bAPs with dSpikes, then a series of bAPs without dSpikes.

File of index entry "branch_adaptation (Fig.4a)": four 500 ms steps. Each epoch's sequence of
failures (F) and dSpikes (D). Criterion: in at least one epoch the sequence is F, D, D, then
only F; the most common sequence is reported.

In [ ]:
import re as _re
rows = []
for rd in D.READINGS:
    s = seq(ocalls[("180146",) + rd])
    rows.append(dict(reading=D.reading_name(*rd), sequences=list(s.values()),
                     epochs_FDD_then_F=sum(bool(_re.fullmatch("FDDF*", x)) for x in s.values()),
                     most_common=pd.Series(list(s.values())).value_counts().index[0]))
print(pd.DataFrame(rows).to_string())

**Result: reproduced.** In all eight readings one of the four step epochs reads failure, dSpike, dSpike, then only failures (FDDFFFFFFFFF), and that is the most common sequence; two other epochs have three or four dSpikes after the first failure and one has none.

## fig4d-example-alternation

> In the Fig. 4c-d neuron, the first 12 bAPs under wide-field stimulation showed two bAP failures
> followed by alternating dSpikes and failures.

File of index entry "branch_bifurcation (Fig.4c,d)": five 600 ms wide-field epochs. Criterion:
the first 12 bAPs read F, F, then D and F alternating, in at least one epoch.

In [ ]:
target = "FF" + "DF" * 5
rows = []
for rd in D.READINGS:
    s = seq(ocalls[("174900",) + rd], 12)
    best = max(s.values(), key=lambda x: sum(a == b for a, b in zip(x, target)))
    rows.append(dict(reading=D.reading_name(*rd), first12=list(s.values()), epochs_matching=sum(x == target for x in s.values()),
                     best_positions_matching=f"{sum(a == b for a, b in zip(best, target))}/12"))
print(pd.DataFrame(rows).to_string())
df = ocalls[("174900", "median", 5.0, np.inf)]
print("bAPs per epoch:", df.groupby("trial").size().tolist(), "; last dSpike (ms after onset) per epoch:",
      df[df.dspike].groupby("trial").t_ms.max().round(0).tolist())

**Result: not reproduced.** Only 0 to 3 bAPs of the 58 to 160 in the five wide-field epochs carry a dSpike, so no epoch shows the alternation; the best epoch matches the target sequence at 8 of 12 positions, mostly through its failures. dSpikes here are called with the same rule as elsewhere; the outputs do not show whether the alternation of Fig. 4d is missed by the rule or absent.

## figs15-stochastic

> Even stronger wide-field stimuli evoked an epoch of seemingly random interplay of bAPs and
> dSpikes, varying from trial to trial while somatic spiking stayed regular, followed by
> sustained dSpike failures.

File of index entry "period_doubling (Fig. S15)": twenty 100 ms stimuli at 5 Hz, each taken as
a trial. Criterion: the dSpike sequence differs between at least two trials, somatic spike
times stay within 2 ms of each other across trials for the same bAP position, and later bAPs
fail.

In [ ]:
rows = []
for rd in D.READINGS:
    df = ocalls[("001459",) + rd]
    s = seq(df)
    sp = df.pivot_table(index="trial", columns="order", values="t_ms")
    n_all = sp.columns[sp.notna().all()]
    spread = (sp[n_all].max() - sp[n_all].min()).round(1)
    last = df.groupby("trial").order.transform("max") == df.order
    rows.append(dict(reading=D.reading_name(*rd), distinct_sequences=len(set(s.values())), sequences=list(s.values())[:20],
                     bAPs_per_trial=df.groupby("trial").size().value_counts().sort_index().to_dict(),
                     spike_time_range_ms_by_position=spread.tolist(), sd_ms=sp[n_all].std().round(2).tolist(),
                     last_bAP_fails=f"{int((~df[last].dspike).sum())}/{int(last.sum())}"))
print(pd.DataFrame(rows).to_string())
h = cells["001459"]["hdr"]
ch = h["daq"]["mod488"]
on, off = D.stim_onsets(h)
tt = ch["t0"] + np.arange(len(ch["x"])) / ch["rate"]
drive = np.array([ch["x"][(tt >= a) & (tt < b)].mean() for a, b in zip(on, off)])
print("488 nm modulation per stimulus (mean DAQ value during the stimulus, V):", np.round(drive, 3).tolist())
print(f"  range {drive.min():.3f} to {drive.max():.3f}; s.d. over mean {drive.std() / drive.mean():.3f};"
      f" stimulus durations {np.round((off - on) * 1000, 1).tolist()} ms")

**Result: not reproduced.** The 488 nm drive is the same in all twenty stimuli (0.200 V, 100 ms each), so they are repeats. Somatic spike times at the same bAP position span 14 to 85 ms across trials (s.d. 2.8 to 24 ms), not within 2 ms, and the number of bAPs per trial varies (2 to 9), so somatic spiking is not regular across trials in this sense. Beyond 300 µm the distal amplitudes have one density mode in every reading (no dSpike threshold); between 300 and 400 µm, 3 to 6 dSpikes are found in three readings, giving 7 or 8 distinct sequences, and every trial ends in failures.

## figs17e-ttx-high-dose

> At 100 nM, TTX eliminated distal dSpikes while somatic spiking continued.

Files of index entries "baseline_TTX (Fig.S17e)" and "after_TTX (100nM; Fig.S17e)" (and 500 nM),
four 1 s ramps each. dSpikes are called with each recording's own trough, and, as a variant,
with the baseline's threshold applied to the TTX recordings (same cell and region). Criterion:
the 100 nM recording has somatic spikes and no distal dSpike; the baseline has dSpikes.

In [ ]:
rows = []
for rd in D.READINGS:
    base = ocalls[("181900",) + rd]
    for key in ["181900", "184232", "185419"]:
        o = cells[key]
        on, off = D.stim_onsets(o["hdr"])
        own = ocalls[(key,) + rd]
        fixed = D.calls(o, on, *rd, window_s=(0.0, float(np.median(off - on))), threshold=base.attrs["dspike_threshold"])
        rows.append(dict(file=key, experiment=o["hdr"]["experiment"], reading=D.reading_name(*rd), bAPs=len(own),
                         dS_own_threshold=int(own.dspike.sum()), dS_baseline_threshold=int(fixed.dspike.sum()),
                         own_thr=round(own.attrs["dspike_threshold"] * 100, 3), base_thr=round(base.attrs["dspike_threshold"] * 100, 3),
                         max_distal=round(own.distal.max() * 100, 3)))
print(pd.DataFrame(rows).to_string())

**Result: blocked (methods underspecified).** The filter readings disagree on whether the baseline has dSpikes at all. With the median filter, and with the Butterworth filter at 4 s.d. between 300 and 400 µm, the baseline's distal amplitudes have one density mode (no trough), so no dSpike can be called (5 of 8 readings). In the other 3 readings (Butterworth) the baseline has 6, 2 and 22 dSpikes, and the 100 nM recording has somatic spikes (1, 1 and 8 bAPs) and no dSpike under the baseline's threshold, as the claim says. Under every reading somatic bAPs fall from 18 to 62 to 1 to 12 at 100 nM and the largest distal amplitude from 1.5 to 3.0% to 0.1 to 0.8%; at 500 nM no somatic spike is found.

## figs18g-ba-increases-calcium-spikes

> Adding Ba2+ (150 µM) increased the incidence of putative dendritic Ca2+ spikes under
> soma-targeted ramp stimulation.

Files of index entries "baseline (Fig.S18g)", "after_Ba2+ (Fig.S18g)" and
"after_Ba2+_NiCl (Fig.S18g)", four 500 ms stimuli each. Putative Ca2+ spikes are distal
depolarizations lasting more than 20 ms (Fig. S18f legend). The distal trace (beyond 300 µm, or
300 to 400 µm) minus a 151 ms moving median is searched for peaks above the file's dSpike
threshold; readings of duration (a) FWHM, (b) time above the dSpike threshold. Counts per
stimulus, mean over the four. Criterion: more after Ba2+ than at baseline, fewer again after
Ni2+.

In [ ]:
from scipy import ndimage, signal

def ca_spikes(key, rd, how, thr=None):
    o = cells[key]
    on, off = D.stim_onsets(o["hdr"])
    thr = ocalls[(key,) + rd].attrs["dspike_threshold"] if thr is None else thr
    if not np.isfinite(thr):
        return [np.nan] * len(on)
    Y = D.bleach_correct(o["X"], o["t"])
    yd, _ = D.distal_trace(Y, o["dist"], 300, rd[2])
    k = int(0.151 * o["fs"]) | 1
    z = yd - ndimage.median_filter(yd, size=k, mode="nearest")
    z = ndimage.uniform_filter1d(z, 3)
    counts = []
    for s, e in zip(on, off):
        i0, i1 = np.searchsorted(o["t"], [s, e + 0.1])
        pk, _ = signal.find_peaks(z[i0:i1], height=thr, distance=int(0.02 * o["fs"]))
        n = 0
        for p in pk:
            if how == "fwhm":
                w = D.fwhm_ms(z[i0:i1], p, o["fs"], max_ms=200)
            else:
                lo = p
                while lo > 0 and z[i0 + lo] > thr:
                    lo -= 1
                hi = p
                while i0 + hi < len(z) - 1 and z[i0 + hi] > thr:
                    hi += 1
                w = (hi - lo) / o["fs"] * 1000
            n += w > 20
        counts.append(n)
    return counts

rows = []
for rd in D.READINGS:
    for how in ("fwhm", "above threshold"):
        c = {key: ca_spikes(key, rd, how) for key in ["145100", "152954", "154649"]}
        m = {key: np.mean(v) for key, v in c.items()}
        rows.append(dict(reading=D.reading_name(*rd), duration=how, baseline=c["145100"], Ba=c["152954"], Ba_Ni=c["154649"],
                         means=[round(v, 2) for v in m.values()],
                         Ba_above_baseline=m["152954"] > m["145100"], Ni_below_Ba=m["154649"] < m["152954"]))
print(pd.DataFrame(rows).to_string())
# One threshold for the three files of the cell: the baseline's, or the Ba2+ file's (as the TTX
# claim uses the baseline's for its drug files).
rows = []
for rd in D.READINGS:
    for src in ("145100", "152954"):
        thr = ocalls[(src,) + rd].attrs["dspike_threshold"]
        for how in ("fwhm", "above threshold"):
            c = {key: ca_spikes(key, rd, how, thr) for key in ["145100", "152954", "154649"]}
            m = {key: np.mean(v) for key, v in c.items()}
            rows.append(dict(reading=D.reading_name(*rd), shared_threshold_of=src, thr_pct=round(thr * 100, 3), duration=how,
                             means=[round(v, 2) for v in m.values()],
                             Ba_above_baseline=m["152954"] > m["145100"], Ni_below_Ba=m["154649"] < m["152954"]))
shared = pd.DataFrame(rows)
print(shared.to_string())
print("readings with a shared threshold where both comparisons hold:", int((shared.Ba_above_baseline & shared.Ni_below_Ba).sum()), "of", len(shared),
      "; with the threshold undefined:", int(shared.thr_pct.isna().sum()))

**Result: reproduced.** With one threshold for the three files of the cell, the baseline's (defined in all 8 readings), every reading and both duration readings give no putative Ca2+ spike at baseline, 1.0 to 2.5 per stimulus after Ba2+, and none after Ba2+ with Ni2+ (16 of 16). With the Ba2+ file's threshold the same holds in 12 of 12 readings where it is defined (4 undefined). With each file's own threshold, the Ni2+ file has no trough in 6 of 8 readings and the Ba2+ file in 2, so that comparison is defined only in 2 Butterworth readings (5 s.d. beyond 300 µm, 4 s.d. between 300 and 400 µm), where it holds (0, 2.0 to 3.75, 0).

## Plateau recordings

Each 4 s plateau file holds three epochs, one trial each: optogenetic stimulation of the soma
alone (30 ms, or 10 ms in the Fig. 5c bottom series), the optogenetic stimulus with EFS
(combined), and EFS alone. The epochs are read from the 488 nm and EFS channels. The distal
trace is the traced trunk beyond 200 µm (Fig. 5c legend); the bleach fit leaves out the 800 ms
after each stimulus.

In [ ]:
def efs_channel(h):
    return [c for c in h["daq"] if c not in ("mod488", "mod594") and h["daq"][c]["x"].max() > h["daq"][c]["x"].min()
            and "enable" not in c.lower() and "shutter" not in c.lower()][0]

def plateau_epochs(key):
    o = cells[key]
    h = o["hdr"]
    opto, opto_off = D.stim_onsets(h)
    efs, _ = D.onsets(h, efs_channel(h))
    ep = []
    used = set()
    for s, e in zip(opto, opto_off):
        near = [x for x in efs if abs(x - s) < 0.3]
        if near:
            ep.append(dict(kind="combined", start=min(s, near[0]), opto=s, opto_end=e, efs=near[0]))
            used.add(near[0])
        else:
            ep.append(dict(kind="optical", start=s, opto=s, opto_end=e, efs=np.nan))
    for x in efs:
        if x not in used:
            ep.append(dict(kind="electrical", start=x, opto=np.nan, opto_end=np.nan, efs=x))
    ep = pd.DataFrame(ep).sort_values("start").reset_index(drop=True)
    starts = ep.start.values
    ep["end"] = [min([v for v in starts if v > s] + [o["t"][-1]]) - 0.05 for s in starts]
    return ep

def plateau_traces(key, lo_um=200.0):
    o = cells[key]
    ep = plateau_epochs(key)
    excl = np.zeros(len(o["t"]), bool)
    for s in ep.start:
        excl |= (o["t"] >= s) & (o["t"] < s + 0.8)
    Y = D.bleach_correct(o["X"], o["t"], exclude=excl)
    yd, n = D.distal_trace(Y, o["dist"], lo_um)
    return Y, yd, ep

for key in ["174622"] + S22 + ["200953", "201133", "201912", "202041"]:
    ep = plateau_epochs(key)
    print(key, cells[key]["hdr"]["experiment"], [(r.kind, round(r.start, 3), round(r.end - r.start, 2)) for r in ep.itertuples()])

## fig5a-plateau-collision

> Combining soma optogenetic stimulation with EFS of distal synaptic inputs evoked a
> long-lasting (about 120 ms) dendritic plateau potential and a burst of somatic spikes, unlike
> either stimulus alone.

File of index entry "Plateau_induction (Fig.5a)". Duration: the time the distal (> 200 µm)
-dF/F, smoothed over 5 ms, stays above half its peak (claims file's standard measure). Somatic
spikes: high-passed somatic trace above 5 (or 4) robust s.d. within 300 ms of the epoch start.
Criterion: combined duration 120 ± 12 ms, more somatic spikes than optical alone, and neither
single stimulus gives a distal event over 60 ms.

In [ ]:
o = cells["174622"]
Y, yd, ep = plateau_traces("174622")
rows = []
for r in ep.itertuples():
    dur, peak = D.above_half_duration_ms(yd, o["t"], r.start, r.end)
    rd_rows = {}
    for rd in ("median", "butter"):
        H = D.highpass(Y[:, 0], o["fs"], rd)
        for k in (5.0, 4.0):
            pk, _ = D.detect_spikes(H, o["fs"], k=k)
            rd_rows[f"spikes_{rd}_k{k:g}"] = int(((o["t"][pk] >= r.start) & (o["t"][pk] < r.start + 0.3)).sum())
    rows.append(dict(epoch=r.kind, start_s=round(r.start, 3), distal_peak_pct=round(peak * 100, 2), duration_ms=round(dur, 1), **rd_rows))
fig5a = pd.DataFrame(rows)
print(fig5a.to_string())
fig, ax = plt.subplots(figsize=(9, 2.8))
for r in ep.itertuples():
    i0, i1 = np.searchsorted(o["t"], [r.start - 0.1, r.start + 0.6])
    ax.plot((o["t"][i0:i1] - r.start) * 1000, yd[i0:i1] * 100, label=r.kind)
ax.set_xlabel("ms after epoch start")
ax.set_ylabel("distal -dF/F (%)")
ax.legend()
plt.show()
print("plotted range (%):", round(float(yd.min() * 100), 2), "to", round(float(yd.max() * 100), 2))

**Result: not reproduced.** The combined stimulus gives a distal (beyond 200 µm) depolarization lasting 146 ms above half its peak, outside 120 ± 12 ms. The other parts of the criterion hold: 9 to 14 somatic spikes in 300 ms against 3 for the optogenetic stimulus alone, and distal events of 9 ms (optical) and 33 ms (EFS) for the single stimuli, under 60 ms.

## fig5d-single-bap-no-plateau

> A single bAP (10 ms soma stimulus) did not evoke a nonlinear dendritic response at any timing
> relative to EFS (5 cells).

The dandiset holds one of the five cells: the four 10 ms files of index entry
"Plateau_varying_timing (Fig.5c)". Per file, AUC combined over AUC optical plus AUC electrical,
distal region beyond 200 µm, window readings (a) until the cumulative area stops rising and (b)
to the end of the epoch. Criterion: no ratio above 2.

In [ ]:
def auc_ratio(key, reading, lo_um=200.0):
    o = cells[key]
    Y, yd, ep = plateau_traces(key, lo_um)
    a = {r.kind: D.auc(yd, o["t"], r.start, r.end, reading) for r in ep.itertuples()}
    return a, a["combined"] / (a["optical"] + a["electrical"])

rows = []
for key in ["200953", "201133", "201912", "202041"]:
    name = D.asset_for(key)["path"].split("/")[0]
    ep = plateau_epochs(key)
    c = ep[ep.kind == "combined"].iloc[0]
    for reading in ("a", "b"):
        a, ratio = auc_ratio(key, reading)
        rows.append(dict(file=name[4:], opto_minus_efs_ms=round((c.opto - c.efs) * 1000), window=reading,
                         auc_opt=round(a["optical"], 3), auc_efs=round(a["electrical"], 3), auc_comb=round(a["combined"], 3),
                         ratio=round(ratio, 2)))
fig5dsb = pd.DataFrame(rows)
print(fig5dsb.to_string())
print("largest ratio by window reading:", fig5dsb.groupby("window").ratio.max().to_dict())

**Result: blocked (methods underspecified).** The two window readings disagree on the one cell of five in the dandiset. Up to the end of the cumulative rise (a), the combined AUC exceeds twice the sum of the single AUCs at +30 ms (2.58) and -100 ms (3.77). To the end of the epoch (b), the single-stimulus AUCs are near zero or negative and no ratio exceeds 2 (-9.9 to 1.59). The criterion makes disagreeing window readings methods_underspecified.

## fig5b-supralinear-auc (blocked: 28 cells, about 10 here)

> The distal AUC for combined stimulation was on average 3.1 ± 0.7-fold the sum of the AUCs for
> optical and EFS stimuli alone, and more than twice that sum in 15 of 28 cells.

The claim needs 28 cells; the dandiset holds the combined-stimulus files of the Fig. 5a and Fig.
S22 examples, and nothing links them to rows of the Source Data. The ratio is computed for them
(both window readings, as fig5d-single-bap-no-plateau) and set beside the Source Data's 28
per-cell values, for comparison only; the claim stays blocked.

In [ ]:
sd5 = pd.read_excel(io.BytesIO(blob), sheet_name="Figure 5", header=None).iloc[3:, :3].dropna().astype(float)
sd5.columns = ["opto_pct", "efs_pct", "comb_pct"]
print(f"Source Data Fig. 5b: {len(sd5)} cells, ratio mean {sd5.comb_pct.mean() / 100:.2f} ± {stats.sem(sd5.comb_pct) / 100:.2f} (s.e.m.),"
      f" above 2 in {int((sd5.comb_pct > 200).sum())}, range {sd5.comb_pct.min() / 100:.2f} to {sd5.comb_pct.max() / 100:.2f};"
      f" optical share of the single-stimulus sum {sd5.opto_pct.min():.0f} to {sd5.opto_pct.max():.0f}%")
rows = []
for key in ["174622"] + S22:
    ep = plateau_epochs(key)
    if not {"optical", "electrical", "combined"} <= set(ep.kind):
        rows.append(dict(file=key, window="-", note="an epoch kind is missing"))
        continue
    c = ep[ep.kind == "combined"].iloc[0]
    for reading in ("a", "b"):
        a, ratio = auc_ratio(key, reading)
        tot = a["optical"] + a["electrical"]
        rows.append(dict(file=key, window=reading, opto_ms=round((c.opto_end - c.opto) * 1000), auc_opt=round(a["optical"], 3),
                         auc_efs=round(a["electrical"], 3), auc_comb=round(a["combined"], 3), ratio=round(ratio, 2),
                         opto_share_pct=round(100 * a["optical"] / tot, 1) if tot else np.nan))
fig5b = pd.DataFrame(rows)
print(fig5b.to_string())
for reading, g in fig5b[fig5b.window.isin(["a", "b"])].groupby("window"):
    ok = g[(g.auc_opt > 0) & (g.auc_efs > 0)]
    print(f"window ({reading}): {len(g)} cells, {len(ok)} with both single-stimulus areas positive; over those, ratio mean"
          f" {ok.ratio.mean():.2f} ± {stats.sem(ok.ratio):.2f} (s.e.m.), above 2 in {int((ok.ratio > 2).sum())},"
          f" inside the Source Data range in {int(ok.ratio.between(sd5.comb_pct.min() / 100, sd5.comb_pct.max() / 100).sum())}")

**Result: blocked (data missing).** The claim needs the 28 cells of Fig. 5b; the dandiset holds 10 combined-stimulus cells with 30 ms soma stimulation, and nothing links them to rows of the Source Data. Over these 10, with the window ending when the cumulative area stops rising (a), the ratio is 3.84 ± 0.96 (mean ± s.e.m.) and above 2 in 7 of 10, against 3.12 ± 0.66 and 15 of 28 in the Source Data; 9 of the 10 fall inside the Source Data's range (0.86 to 16.29). With the window to the end of the epoch (b) one or both single-stimulus areas are negative in 7 of 10 cells, so the ratio is undefined there.

## figs23-dendrite-drives-soma

> Subtracting the somatic signal from the dendritic signal reveals a slowly rising and falling
> dendritic depolarization lasting up to about 300 ms after the optogenetic stimulus, which
> drives the high-frequency somatic firing.

The nine files of index entry "Plateau (Fig.S22)" (the gallery of Fig. S23 shows 9 neurons).
In the combined epoch (the first one where a file has several), the dendritic region is the 5
trunk segments (about 33 µm, the Methods' example width) with the largest mean -dF/F from 30 to
300 ms after the start; the somatic -dF/F is subtracted from it. Persistence: the time after
the 30 ms optogenetic stimulus ends until the difference, smoothed over 5 ms, returns within 2
s.d. of its 200 ms pre-stimulus baseline. Criterion: the longest persistence over cells is
300 ± 30 ms and the difference is positive in most cells (mean over 30 to 300 ms).

In [ ]:
def figs23_row(key, lo_um=0.0, smooth_ms=5.0):
    o = cells[key]
    ep = plateau_epochs(key)
    excl = np.zeros(len(o["t"]), bool)
    for s in ep.start:
        excl |= (o["t"] >= s) & (o["t"] < s + 0.8)
    Y = D.bleach_correct(o["X"], o["t"], exclude=excl)
    c = ep[ep.kind == "combined"].iloc[0]
    t = o["t"]
    i0, i30, i300 = np.searchsorted(t, [c.start, c.start + 0.03, c.start + 0.3])
    seg = Y[:, 1:-1]
    sm = ndimage.uniform_filter1d(seg, 5, axis=1)  # mean of 5 neighbouring segments around each
    cand = [i for i in range(2, seg.shape[1] - 2) if o["dist"][i - 2] > lo_um]
    best = cand[int(np.argmax(sm[i30:i300][:, cand].mean(0)))]
    dend = seg[:, best - 2:best + 3].mean(1)
    diff = ndimage.uniform_filter1d(dend - Y[:, 0], max(1, int(smooth_ms / 1000 * o["fs"])))
    b0 = np.searchsorted(t, c.start - 0.2)
    base = diff[b0:i0]
    j = np.searchsorted(t, c.opto_end)
    jend = np.searchsorted(t, c.end)
    inside = np.abs(diff[j:jend] - base.mean()) <= 2 * base.std()
    ret = int(np.argmax(inside)) if inside.any() else jend - j
    row = dict(file=key, region=f">{lo_um:g} um", smooth_ms=smooth_ms, dendrite_um=round(float(o["dist"][best]), 0),
               mean_diff_30_300ms_pct=round(float((diff[i30:i300] - base.mean()).mean() * 100), 2),
               persistence_ms=round(ret / o["fs"] * 1000, 1), epoch_left_ms=round((jend - j) / o["fs"] * 1000))
    return row, (t[b0:jend] - c.opto_end, (diff[b0:jend] - base.mean()) * 100, 2 * base.std() * 100)

rows, traces = [], {}
for lo_um in (0.0, 200.0):
    for smooth_ms in (5.0, 20.0):
        for key in S22:
            row, tr = figs23_row(key, lo_um, smooth_ms)
            rows.append(row)
            traces[(key, lo_um, smooth_ms)] = tr
figs23 = pd.DataFrame(rows)
print(figs23.to_string())
for (region, sm_), g in figs23.groupby(["region", "smooth_ms"]):
    print(f"region {region}, smoothing {sm_:g} ms: longest persistence {g.persistence_ms.max()} ms, median {g.persistence_ms.median()} ms;"
          f" cells with a positive difference {int((g.mean_diff_30_300ms_pct > 0).sum())} of {len(g)}")
fig, axes = plt.subplots(1, len(S22), figsize=(2.2 * len(S22), 2.6), sharex=True)
for ax, key in zip(axes, S22):
    x, y, band = traces[(key, 200.0, 20.0)]
    ax.plot(x * 1000, y, lw=0.7)
    ax.axhspan(-band, band, color="0.85")
    ax.axvline(0, color="k", lw=0.5)
    ax.set_title(key, fontsize=8)
    ax.set_xlabel("ms after stimulus end")
axes[0].set_ylabel("dendrite - soma -dF/F (%)")
plt.tight_layout()
plt.show()
print("plotted range (%):", round(min(float(traces[(k, 200.0, 20.0)][1].min()) for k in S22), 2), "to",
      round(max(float(traces[(k, 200.0, 20.0)][1].max()) for k in S22), 2))

**Result: not reproduced.** The dendrite-minus-soma -dF/F is positive from 30 to 300 ms in 8 of 9 cells (7 of 9 with the region beyond 200 µm), but the longest persistence after the stimulus ends is 206 ms (5 ms smoothing) or 222 ms (20 ms smoothing) with the region chosen anywhere on the trunk, and 170 or 220 ms beyond 200 µm, short of 270 to 330 ms in every reading. The 20 ms smoothing removes most of the noise-driven early returns (median persistence 58 to 99 ms over the trunk, 19 to 85 ms beyond 200 µm), but cell 160204, whose mean difference is +1.3 to +2.7%, still returns within 2 s.d. at once (0 to 6 ms) although its mean difference is positive, so the persistence measure stays sensitive to the first frames after the stimulus.

## Patch clamp with imaging (Fig. S2)

The 14 files of index entries "Patch clamp (Fig. S2a/b)", "Patch clamp_1kHz (Fig. S2d)" and
"Patch clamp_2kHz (Fig. S2d)": seven cells, each imaged at 1 kHz and at 2 kHz (pairs matched by
the slice and cell in the file name) with simultaneous current clamp. The files hold the
amplifier's primary output; the dandiset description gives its gain (50 mV/V, so -1.5 V is -75
mV). Fluorescence is the mean of the cell's pixels (the brightest 20% of the mean image), as in
the Methods' unweighted region. Each frame's voltage is the mean of the 10 kHz samples within
the frame. The bleach fit uses frames at rest (below -60 mV).

In [ ]:
PATCH = {"V2-11": ("000919", "000820"), "V2-14": ("150706", "150945"), "Cel3": ("205240", "205908"),
         "V2-10": ("232204", "233122"), "Cell1-ctx": ("215254", "215422"), "V2-12": ("004720", "004921"),
         "Cell4": ("212457", "212620")}
MV_PER_V = 50.0  # dandiset description

def patch_record(key):
    h = D.read_header(key)
    m = D.mean_image(h)
    W = D.weights([m > np.percentile(m, 80)])
    F = D.extract(h, W)[:, 0]
    t = h["cam_t"]
    v = h["daq"]["Primary_out"]
    vt = v["t0"] + np.arange(len(v["x"])) / v["rate"]
    dt = np.median(np.diff(t))
    idx = np.searchsorted(vt, t)
    step = int(round(dt * v["rate"]))
    cs = np.r_[0, np.cumsum(v["x"].astype(np.float64))]
    hi = np.minimum(idx + step, len(v["x"]))
    vf = (cs[hi] - cs[idx]) / np.maximum(hi - idx, 1) * MV_PER_V
    return dict(key=key, t=t, F=F, vf=vf, vt=vt, v=v["x"] * MV_PER_V, fs=1 / dt, path=h["path"])

t0 = time.time()
precs = {key: patch_record(key) for pair in PATCH.values() for key in pair}
print(f"loaded {len(precs)} patch files in {time.time() - t0:.0f} s")

In [ ]:
rows = []
for cell, (k1, k2) in PATCH.items():
    for rate, key in (("1 kHz", k1), ("2 kHz", k2)):
        r = precs[key]
        rows.append(dict(cell=cell, rate=rate, file=key, frames=len(r["t"]), frame_ms=round(1000 / r["fs"], 3),
                         vm_min=round(float(r["v"].min()), 1), vm_max=round(float(r["v"].max()), 1),
                         vm_rest_median=round(float(np.median(r["v"][:5000])), 1), F_min=int(r["F"].min()), F_max=int(r["F"].max())))
print(pd.DataFrame(rows).to_string())

Membrane potential spans about -75 to +45 mV with the stated gain, which fits a CA1 neuron at
rest and at spike peaks; the gain is consistent with the data. Clock check: optical and
electrical spikes are matched below; the median lag between them shows whether the camera and
DAQ clocks agree.

## figs2-imaging-fidelity

> Optical recording at 1 kHz detected every spike up to 90 Hz with no false positives or
> negatives (7 cells, 2212 spikes), with a single-trial noise-equivalent voltage of 1.2 ± 0.5 mV
> (2.5 ± 0.8 mV at 2 kHz) and sensitivity of about 12% dF/F per 100 mV.

Sensitivity: slope of -dF/F against the frame voltage, over subthreshold frames (below -40 mV and
more than 5 ms from any spike), times 100 mV. Noise-equivalent voltage, readings (a) the s.d. of
-dF/F over the first second (at rest, no current) divided by the slope, (b) the s.d. of the
optical trace scaled to mV minus the frame voltage, over subthreshold frames. Electrical spikes:
upward crossings of 0 mV, at least 3 ms apart. Optical spikes: high-passed (20 ms median) -dF/F above 5 robust s.d.,
peaks at least 3 ms apart. A pair matches when the optical peak falls within one frame of the
electrical spike, after the median lag is removed.

In [ ]:
def patch_measures(r, k=5.0, repol_mV=None):
    t, F, vf = r["t"], r["F"], r["vf"]
    rest = vf < -60
    y = D.bleach_correct(F, t, exclude=~rest)
    vx = r["v"]
    up = np.flatnonzero((vx[1:] >= 0) & (vx[:-1] < 0)) + 1
    keep = []
    for u in up:  # one spike per crossing; crossings within 3 ms of the last kept one are noise
        if keep and r["vt"][u] - r["vt"][keep[-1]] <= 0.003:
            continue
        if repol_mV is not None and keep and vx[keep[-1]:u].min() > repol_mV:
            continue  # variant: a new spike only after the membrane repolarized below repol_mV
        keep.append(u)
    espk = r["vt"][np.array(keep, int)]
    near = np.zeros(len(t), bool)
    for s in espk:
        near |= np.abs(t - s) < 0.005
    sub = (vf < -40) & ~near
    slope, icpt = np.polyfit(vf[sub], y[sub], 1)
    q = t < 1.0
    nev_a = y[q].std() / slope
    pred = (y - icpt) / slope
    nev_b = (pred[sub] - vf[sub]).std()
    H = D.highpass(y, r["fs"], "median")
    pk, _ = D.detect_spikes(H, r["fs"], k=k)
    ospk = t[pk]
    lags = np.array([ospk[np.argmin(np.abs(ospk - s))] - s for s in espk]) if len(ospk) else np.array([])
    lag = float(np.median(lags)) if len(lags) else 0.0
    frame = 1 / r["fs"]
    used = np.zeros(len(ospk), bool)
    hits = 0
    for s in espk:
        d = np.abs(ospk - lag - s)
        j = int(np.argmin(d)) if len(d) else -1
        if j >= 0 and d[j] <= frame and not used[j]:
            used[j] = True
            hits += 1
    isi = np.diff(espk)
    return dict(sens_pct_per_100mV=round(slope * 100 * 100, 2), dFF_at_20mV_pct=round(slope * 90 * 100, 2),
                nev_a_mV=round(nev_a, 2), nev_b_mV=round(nev_b, 2), e_spikes=len(espk), o_spikes=len(ospk),
                matched=hits, false_neg=len(espk) - hits, false_pos=int((~used).sum()), lag_ms=round(lag * 1000, 2),
                max_rate_Hz=round(1 / isi.min(), 1) if len(isi) else np.nan)

rows = []
for cell, (k1, k2) in PATCH.items():
    for rate, key in (("1 kHz", k1), ("2 kHz", k2)):
        for k, repol in ((5.0, np.nan), (4.0, np.nan), (5.0, -30.0)):
            rows.append(dict(cell=cell, rate=rate, k=k, repol_mV=repol,
                             **patch_measures(precs[key], k, None if np.isnan(repol) else repol)))
s2 = pd.DataFrame(rows)
print(s2.to_string())
for (k, repol), g0 in s2.groupby(["k", "repol_mV"], dropna=False):
    g = g0.groupby("rate")
    print(f"k={k:g}, electrical spikes {'after repolarization below ' + str(repol) + ' mV' if np.isfinite(repol) else '3 ms apart'}:",
          g[["sens_pct_per_100mV", "nev_a_mV", "nev_b_mV", "max_rate_Hz"]].agg(["mean", "std"]).round(2).to_string())
    print("   spikes (electrical), false negatives, false positives by rate:",
          g[["e_spikes", "false_neg", "false_pos"]].sum().to_dict("index"))

**Result: not reproduced.** Sensitivity is 11.7 ± 2.9% per 100 mV at 1 kHz (10.8 ± 3.5% at 2 kHz), within 12 ± 1.2 on average. The noise-equivalent voltage is 3.5 ± 0.5 mV (reading a) and 3.3 ± 0.9 mV (b) at 1 kHz, against 1.2 ± 0.5, and 11.4 and 5.2 mV at 2 kHz, against 2.5 ± 0.8. At 1 kHz, 6 of 1106 electrical spikes have no optical match and 19 optical spikes have no electrical one (5 s.d.; 46 with 4 s.d.). Counting an electrical spike only after the membrane repolarized below -30 mV removes the impossible rates (largest 91 Hz; 333 and 263 Hz before) and the extra electrical spikes, and gives 1105 spikes with 5 missed and 19 added at 1 kHz, and 1107 with 189 missed and 372 added at 2 kHz, so the false detections are not an artefact of the electrical count. The NEV does not depend on that count.

## figs2d-frame-rate-same-sensitivity

> Fluorescence changes for a step from -70 to +20 mV were the same at 1 kHz and 2 kHz.

The files hold current-clamp steps and ramps, not a voltage step, so the change for -70 to +20
mV is read from each cell's fluorescence-voltage relation (Fig. S2d left): the subthreshold slope
times 90 mV. Two-sided paired t-test over the seven cells at 0.05. The Source Data's per-cell
values (sheet Supplementary Fig 2) are printed beside, for comparison only.

In [ ]:
w = s2[(s2.k == 5.0) & s2.repol_mV.isna()].pivot(index="cell", columns="rate", values="dFF_at_20mV_pct")
print(w.round(2).to_string())
tt = stats.ttest_rel(w["1 kHz"], w["2 kHz"])
print(f"paired t-test: t = {tt.statistic:.2f}, p = {tt.pvalue:.3f}; mean difference (1 - 2 kHz) = {(w['1 kHz'] - w['2 kHz']).mean():.2f} percentage points")
sd2 = pd.read_excel(io.BytesIO(blob), sheet_name="Supplementary Fig 2", header=None).iloc[3:10, :2].astype(float)
sd2.columns = ["1 kHz", "2 kHz"]
print("Source Data, -dF/F at 20 mV (%): 1 kHz mean", round(sd2["1 kHz"].mean(), 2), "2 kHz mean", round(sd2["2 kHz"].mean(), 2),
      "; paired t-test p =", round(stats.ttest_rel(sd2["1 kHz"], sd2["2 kHz"]).pvalue, 3))

**Result: reproduced.** The -dF/F change from -70 to +20 mV, read from each cell's subthreshold fluorescence-voltage slope, does not differ between 1 kHz and 2 kHz (paired t-test p = 0.14, mean difference 0.84 percentage points, 7 cells). Per-cell values here range from 6.1 to 16.0%; the Source Data's means are 10.85% (1 kHz) and 10.93% (2 kHz).

## Summary

| Claim | Outcome |
|---|---|
| source-data-index | reproduced |
| fig2a-dspike-dichotomy | reproduced |
| fig2b-bimodal-distal-amplitudes | blocked, methods underspecified (the two modality readings disagree) |
| fig2e-f-dspike-maps | not reproduced |
| fig2h-length-constant | not reproduced |
| fig3-subthreshold-no-dendritic-initiation | not reproduced (one or two small isolated distal events) |
| fig3-two-motifs-location-independent | not reproduced |
| fig3e-soma-third-bap | not reproduced |
| fig3d-distal-first-bap | reproduced, on an assumption (4 s.d. somatic threshold) |
| fig3f-soma-plus-distal | not reproduced (46 of 54 beyond 300 µm; 49 to 51 between 300 and 400 µm) |
| fig3-predepolarization-rule | not reproduced |
| fig1-denoising-variance | blocked, methods underspecified (the share depends on the preprocessing) |
| fig4a-step-example | reproduced |
| fig4d-example-alternation | not reproduced |
| figs15-stochastic | not reproduced |
| figs17e-ttx-high-dose | blocked, methods underspecified (the filter readings disagree on baseline dSpikes) |
| figs18g-ba-increases-calcium-spikes | reproduced |
| fig5a-plateau-collision | not reproduced (duration 146 ms) |
| fig5b-supralinear-auc | blocked, data missing (10 of 28 cells, shown beside the Source Data) |
| fig5d-single-bap-no-plateau | blocked, methods underspecified (the window readings disagree; 1 of 5 cells) |
| figs23-dendrite-drives-soma | not reproduced |
| figs2-imaging-fidelity | not reproduced |
| figs2d-frame-rate-same-sensitivity | reproduced |
| 28 other claims on pooled samples | blocked, data missing |
| fig4f-g-model-accelerometer (simulations notebook) | not reproduced |
| figs20ab-model-patch-vs-opto (simulations notebook) | reproduced |
| figs20-model-robustness (simulations notebook) | not reproduced |
| figs21-izhikevich (simulations notebook, Python port) | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The dandiset holds raw movies only, so every
region is found here: the soma and a line along the trunk traced on the mean image (the paper
drew them by hand on a structural image), with the Methods' 0.65 µm per pixel. The figure under "Where the regions lie" shows that the traced line leaves the trunk beyond about 400 µm (the mean image along it falls below the image median). Spikes are found
with the claims file's rules (high-pass filter readings, a robust-s.d. somatic threshold, a
trough threshold for dSpikes with FWHM < 5 ms), where the paper set its thresholds by hand. Where the distal amplitudes have a single density mode no dSpike threshold exists, and the dSpike count of that recording is treated as undefined, not as zero. The
claims file's somatic threshold of 5 robust s.d. misses closely spaced spikes in the burst files
(150 bAPs where the legend counts 167 and 170); 4 s.d. gives the legend's totals and is adopted
for the claims it changes, with 5 s.d. shown. The traced line runs off the trunk beyond about 400
µm, so a distal region of 300 to 400 µm is shown beside the claims file's "beyond 300 µm". The
dF/F_ref reference (10 to 20 ms after a spike) carries the stimulus-driven depolarization when
bAPs come in bursts, so raw -dF/F is shown beside it. All results except the Fig. S2 ones rest
on one neuron or one example recording per claim.

What was found about the dataset. The 84 files are the example recordings of the Source
Data index; the pooled samples behind most statistics are not deposited. Three recordings are
deposited twice. The files carry no pixel size, regions, structural images or stimulus masks;
the subject of every file is the file itself (81 subjects, date of birth 1900-01-01, sex
unknown), so animals and cells cannot be told apart except through file names. Frames are 1.03
ms apart (971 Hz) where the paper states 1 kHz, and 0.52 ms (1923 Hz) where it states 2 kHz. The
Fig. 3 legend's D2 is the file named D4, and the Fig. 2 legend's "D1 stim" is the file named D3.